In [ ]:
!pip install langchain langchain-community langchain-google-genai pageindex rank-bm25

In [ ]:
import os
import time
import json
import re
import string
from google.colab import userdata

os.environ["GOOGLE_API_KEY"] = userdata.get('GOOGLE_API_KEY')

from pageindex import PageIndexClient
pi_client = PageIndexClient(api_key=userdata.get('PAGEINDEX'))

## Corpus Preparation

In [ ]:
import nltk
from nltk.corpus import stopwords
from langchain.text_splitter import RecursiveCharacterTextSplitter

nltk.download('stopwords')
nltk.download('punkt')

stop_words = set(stopwords.words('english'))

def clean_text(text):
    text = text.lower()
    text = text.translate(str.maketrans('', '', string.punctuation))
    tokens = text.split()
    tokens = [t for t in tokens if t not in stop_words]
    return ' '.join(tokens)

with open('corpus.json', 'r') as f:
    raw_articles = json.load(f)

splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)

cleaned_chunks = []
for article in raw_articles:
    cleaned = clean_text(article['text'])
    splits = splitter.split_text(cleaned)
    for chunk in splits:
        cleaned_chunks.append({'text': chunk, 'source': article.get('url', '')})

print(f"Total chunks: {len(cleaned_chunks)}")

## Hierarchical PageIndex

In [ ]:
corpus_text = '\n\n'.join([c['text'] for c in cleaned_chunks])

with open('corpus_for_pageindex.txt', 'w') as f:
    f.write(corpus_text)

In [ ]:
with open('corpus_for_pageindex.txt', 'rb') as f:
    response = pi_client.upload(file=f, filename='corpus_for_pageindex.txt')

doc_id = response['id']
print(f"Uploaded. Document ID: {doc_id}")

while True:
    status = pi_client.get_status(doc_id)
    if status['status'] == 'ready':
        print("Indexing complete.")
        break
    print("Still indexing...")
    time.sleep(10)

In [ ]:
tree = pi_client.get_tree(doc_id)

pageindex_docs = []

def traverse(node, parent_title=''):
    title = node.get('title', parent_title)
    if 'text' in node and node['text']:
        pageindex_docs.append({
            'text': node['text'],
            'title': title,
            'level': node.get('level', 0)
        })
    for child in node.get('children', []):
        traverse(child, title)

traverse(tree)
print(f"Extracted {len(pageindex_docs)} nodes from PageIndex tree.")

## BM25 Retrieval

In [ ]:
from rank_bm25 import BM25Okapi
from langchain.schema import BaseRetriever, Document
from pydantic import Field
from typing import List

tokenized_corpus = [doc['text'].split() for doc in pageindex_docs]
bm25_index = BM25Okapi(tokenized_corpus)

lc_docs = [
    Document(page_content=doc['text'], metadata={'title': doc['title'], 'level': doc['level']})
    for doc in pageindex_docs
]

class BM25Retriever(BaseRetriever):
    bm25: object = Field(default=None)
    docs: List[Document] = Field(default_factory=list)
    k: int = Field(default=5)

    def _get_relevant_documents(self, query: str) -> List[Document]:
        tokens = query.lower().split()
        scores = self.bm25.get_scores(tokens)
        top_indices = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)[:self.k]
        return [self.docs[i] for i in top_indices]

    async def _aget_relevant_documents(self, query: str) -> List[Document]:
        return self._get_relevant_documents(query)

retriever = BM25Retriever(bm25=bm25_index, docs=lc_docs, k=5)
print("BM25 retriever ready.")

## LangChain Integration

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.prompts import ChatPromptTemplate
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain

llm = ChatGoogleGenerativeAI(model="gemini-pro", temperature=0)

prompt = ChatPromptTemplate.from_template("""
Answer the question using only the context provided below. 
If the answer is not found in the context, say "I don't know based on the provided context."
Do not use any outside knowledge.

Context:
{context}

Question: {input}
""")

doc_chain = create_stuff_documents_chain(llm, prompt)
rag_chain = create_retrieval_chain(retriever, doc_chain)

print("RAG chain ready.")

In [ ]:
test_query = "What is the main topic discussed in the corpus?"

result = rag_chain.invoke({"input": test_query})
print("Answer:", result['answer'])

## Latency & Accuracy Comparison

In [ ]:
from langchain.schema import HumanMessage, SystemMessage

test_questions = [
    "What is the main topic discussed in the corpus?",
    "What are the key findings mentioned?",
    "Who are the main entities referenced?"
]

full_context = '\n\n'.join([doc['text'] for doc in pageindex_docs])

results = []

for q in test_questions:
    t0 = time.time()
    bm25_result = rag_chain.invoke({"input": q})
    bm25_time = time.time() - t0

    naive_prompt = f"""Answer the question using only the context below.

Context:
{full_context[:12000]}

Question: {q}"""

    t0 = time.time()
    naive_result = llm.invoke([HumanMessage(content=naive_prompt)])
    naive_time = time.time() - t0

    results.append({
        'question': q,
        'bm25_answer': bm25_result['answer'],
        'bm25_latency': round(bm25_time, 2),
        'naive_answer': naive_result.content,
        'naive_latency': round(naive_time, 2)
    })

for r in results:
    print(f"\nQuestion: {r['question']}")
    print(f"  BM25 ({r['bm25_latency']}s): {r['bm25_answer'][:200]}")
    print(f"  Naive ({r['naive_latency']}s): {r['naive_answer'][:200]}")

In [ ]:
import statistics

bm25_latencies = [r['bm25_latency'] for r in results]
naive_latencies = [r['naive_latency'] for r in results]

print(f"BM25 avg latency: {statistics.mean(bm25_latencies):.2f}s")
print(f"Naive avg latency: {statistics.mean(naive_latencies):.2f}s")
print(f"Speedup factor: {statistics.mean(naive_latencies) / statistics.mean(bm25_latencies):.2f}x")